### RAG Pipelines- Data Ingestion to Vector DB Pipeline

In [5]:
import os
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

In [7]:
### Read all the pdf's inside the directory
def process_all_pdfs(pdf_directory):
    """Process all PDF files in a directory"""
    all_documents = []
    pdf_dir = Path(pdf_directory)
    
    # Find all PDF files recursively
    pdf_files = list(pdf_dir.glob("**/*.pdf"))
    
    print(f"Found {len(pdf_files)} PDF files to process")
    
    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()
            
            # Add source information to metadata
            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'
            
            all_documents.extend(documents)
            print(f"  ✓ Loaded {len(documents)} pages")
            
        except Exception as e:
            print(f"  ✗ Error: {e}")
    
    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents

# Process all PDFs in the data directory
all_pdf_documents = process_all_pdfs("../data")

Found 1 PDF files to process

Processing: IIT2022209_report.pdf
  ✓ Loaded 5 pages

Total documents loaded: 5


In [8]:
all_pdf_documents

[Document(metadata={'producer': 'Skia/PDF m145 Google Docs Renderer', 'creator': 'PyPDF', 'creationdate': '', 'title': 'Untitled document', 'source': '../data/pdf/IIT2022209_report.pdf', 'total_pages': 5, 'page': 0, 'page_label': '1', 'source_file': 'IIT2022209_report.pdf', 'file_type': 'pdf'}, page_content='SUMMER\n \nINTERNSHIP\n \nREPORT\n \n \n \n \nDesign\n \nand\n \nImplementation\n \nof\n \nMachine\n \nLearning\n \nFrameworks\n \nfor\n \nAutonomous\n \nAgricultural\n \nSystems\n \n \nSubmitted\n \nby:\n \nNaveen\n \nTanniru\n \nEnrollment\n \nNo\n \n:\n \nIIT2022209\n \nInstitution:\n \nIndian\n \nInstitute\n \nof\n \nInformation\n \nTechnology\n  \nAllahabad\n \nInternship\n \nOrganization:\n \nBharat\n \nRobotiX\n \n(DEEM\n \nAGROTECH\n \nSOLUTIONS\n \nPRIVATE\n \nLIMITED)\n \nInternship\n \nDuration:\n \nMay\n  \n2025\n \n–\n \nJuly\n  \n2025\n \n \nInternship\n \nType\n \n:\n \nRemote'),
 Document(metadata={'producer': 'Skia/PDF m145 Google Docs Renderer', 'creator': 'PyPDF'

In [9]:
### Text splitting get into chunks

def split_documents(documents,chunk_size=1000,chunk_overlap=200):
    """Split documents into smaller chunks for better RAG performance"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")
    
    # Show example of a chunk
    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")
    
    return split_docs

In [ ]:
chunks=split_documents(all_pdf_documents)
chunks

### embedding And vectorStoreDB

In [11]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

/Users/naveen_tanniru/Desktop/Agentic_AI/RAG/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W1030 22:10:38.345000 61943 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


In [12]:
class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer"""
    
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """
        Initialize the embedding manager
        
        Args:
            model_name: HuggingFace model name for sentence embeddings
        """
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """Load the SentenceTransformer model"""
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
        Generate embeddings for a list of texts
        
        Args:
            texts: List of text strings to embed
            
        Returns:
            numpy array of embeddings with shape (len(texts), embedding_dim)
        """
        if not self.model:
            raise ValueError("Model not loaded")
        
        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings
    
    
## initialize the embedding manager

embedding_manager=EmbeddingManager()
embedding_manager

Loading embedding model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 9967.08it/s]


Model loaded successfully. Embedding dimension: 384


/var/folders/tk/z8yt_k4d5nd5zmr1yysc1jrw0000gn/T/ipykernel_61943/3620522432.py:20: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")


### VectorStore

In [13]:
class VectorStore:
    """Manages document embeddings in a ChromaDB vector store"""
    
    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        """
        Initialize the vector store
        
        Args:
            collection_name: Name of the ChromaDB collection
            persist_directory: Directory to persist the vector store
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """Initialize ChromaDB client and collection"""
        try:
            # Create persistent ChromaDB client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)
            
            # Get or create collection
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "PDF document embeddings for RAG"}
            )
            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise
    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
        Add documents and their embeddings to the vector store
        
        Args:
            documents: List of LangChain documents
            embeddings: Corresponding embeddings for the documents
        """
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")
        
        print(f"Adding {len(documents)} documents to vector store...")
        
        # Prepare data for ChromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []
        
        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # Generate unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)
            
            # Prepare metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)
            
            # Document content
            documents_text.append(doc.page_content)
            
            # Embedding
            embeddings_list.append(embedding.tolist())
        
        # Add to collection
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )
            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise

vectorstore=VectorStore()
vectorstore

Vector store initialized. Collection: pdf_documents
Existing documents in collection: 0


In [14]:
chunks

[Document(metadata={'producer': 'Skia/PDF m145 Google Docs Renderer', 'creator': 'PyPDF', 'creationdate': '', 'title': 'Untitled document', 'source': '../data/pdf/IIT2022209_report.pdf', 'total_pages': 5, 'page': 0, 'page_label': '1', 'source_file': 'IIT2022209_report.pdf', 'file_type': 'pdf'}, page_content='SUMMER\n \nINTERNSHIP\n \nREPORT\n \n \n \n \nDesign\n \nand\n \nImplementation\n \nof\n \nMachine\n \nLearning\n \nFrameworks\n \nfor\n \nAutonomous\n \nAgricultural\n \nSystems\n \n \nSubmitted\n \nby:\n \nNaveen\n \nTanniru\n \nEnrollment\n \nNo\n \n:\n \nIIT2022209\n \nInstitution:\n \nIndian\n \nInstitute\n \nof\n \nInformation\n \nTechnology\n  \nAllahabad\n \nInternship\n \nOrganization:\n \nBharat\n \nRobotiX\n \n(DEEM\n \nAGROTECH\n \nSOLUTIONS\n \nPRIVATE\n \nLIMITED)\n \nInternship\n \nDuration:\n \nMay\n  \n2025\n \n–\n \nJuly\n  \n2025\n \n \nInternship\n \nType\n \n:\n \nRemote'),
 Document(metadata={'producer': 'Skia/PDF m145 Google Docs Renderer', 'creator': 'PyPDF'

In [15]:
### Convert the text to embeddings
texts=[doc.page_content for doc in chunks]

## Generate the Embeddings

embeddings=embedding_manager.generate_embeddings(texts)

##store int he vector dtaabase
vectorstore.add_documents(chunks,embeddings)

Generating embeddings for 13 texts...


Batches: 100%|██████████| 1/1 [00:02<00:00,  2.38s/it]

Generated embeddings with shape: (13, 384)
Adding 13 documents to vector store...
Successfully added 13 documents to vector store
Total documents in collection: 13


In [16]:
class RAGRetriever:
    """Handles query-based retrieval from the vector store"""
    
    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        """
        Initialize the retriever
        
        Args:
            vector_store: Vector store containing document embeddings
            embedding_manager: Manager for generating query embeddings
        """
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        """
        Retrieve relevant documents for a query
        
        Args:
            query: The search query
            top_k: Number of top results to return
            score_threshold: Minimum similarity score threshold
            
        Returns:
            List of dictionaries containing retrieved documents and metadata
        """
        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, Score threshold: {score_threshold}")
        
        # Generate query embedding
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]
        
        # Search in vector store
        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )
            
            # Process results
            retrieved_docs = []
            
            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]
                
                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    # Convert distance to similarity score (ChromaDB uses cosine distance)
                    similarity_score = 1 - distance
                    
                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            'id': doc_id,
                            'content': document,
                            'metadata': metadata,
                            'similarity_score': similarity_score,
                            'distance': distance,
                            'rank': i + 1
                        })
                
                print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
            else:
                print("No documents found")
            
            return retrieved_docs
            
        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []

rag_retriever=RAGRetriever(vectorstore,embedding_manager)

In [17]:
rag_retriever

In [24]:
rag_retriever.retrieve("what is Time-Series Forecasting for Predictive Resource Management")

Retrieving documents for query: 'what is Time-Series Forecasting for Predictive Resource Management'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:01<00:00,  1.69s/it]

Generated embeddings with shape: (1, 384)
Retrieved 1 documents (after filtering)


[{'id': 'doc_f68e4ed1_7',
  'content': 'Project\n \n2:\n \nTime-Series\n \nForecasting\n \nfor\n \nPredictive\n \nResource\n \nManagement\n \nObjective:\n \nTo\n \nbuild\n \na\n \npredictive\n \nmodel\n \nthat\n \nanticipates\n \nfuture\n \nsoil\n \nmoisture\n \ntrends\n \nto\n \noptimize\n \nirrigation\n \nefficiency.\n \nWorkflow:\n \nCurated\n \na\n \ntemporal\n \ndataset\n \nby\n \nmerging\n \nreal-time\n \nsensor\n \nstreams\n \nwith\n \nhistorical\n \n5-year\n \nweather\n \npatterns\n \nfetched\n \nfrom\n \nthe\n \nOpenWeather\n \nAPI.\n \nDeveloped\n \nand\n \ntrained\n \na\n \nLong\n \nShort-Term\n \nMemory\n \n(LSTM)\n \nneural\n \nnetwork\n \narchitecture\n \nto\n \ncapture\n \nseasonal\n \nand\n \ndaily\n \nnon-linear\n \npatterns\n \nin\n \nhumidity\n \nand\n \ntemperature.\n \nConducted\n \nrigorous\n \nHyperparameter\n \nTuning\n \nof\n \nthe\n \nwindow\n \nsize\n \nand\n \ndropout\n \nrates\n \nto\n \nprevent\n \nthe\n \nmodel\n \nfrom\n \noverfitting\n \nto\n \nextreme\

### Integration Vectordb Context pipeline With LLM output

In [26]:
### simple RAG pipeline with Groq LLM 
from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv
load_dotenv()

### Inititalize the groq llm 

llm = ChatGroq(model="qwen/qwen3.8-27b",temperature=0.1,max_tokens=1000)

# simple rag : retrieve context and generate response
def rag_simple(query,retriever,llm,top_k=3):
    # retriever the context
    results=retriever.retrieve(query,top_k=top_k)
    context="\n\n".join([doc['content']for doc in results]) if results else ""
    if not context:
        return "No relevant context found to answer the question."
    #generate the answer using groq llm
    prompt= f"""use the following context to answer the question concisely.
        context:
        {context}
        
        question :{query}
        
        Answer:
        """
    
    response = llm.invoke([prompt.format(context=context,query=query)])
    
    return response.content


In [32]:
answer = rag_simple("what is Time-Series Forecasting for Predictive Resource Management?",rag_retriever,llm)
print(answer)

Retrieving documents for query: 'what is Time-Series Forecasting for Predictive Resource Management?'
Top K: 3, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00,  2.89it/s]


Generated embeddings with shape: (1, 384)
Retrieved 1 documents (after filtering)
It is a project aimed at building a predictive model to anticipate future soil moisture trends, thereby optimizing irrigation efficiency.


### Enhanced RAG Pipeline

In [ ]:
# --- Enhanced RAG Pipeline Features ---
def rag_advanced(query, retriever, llm, top_k=5, min_score=0.2, return_context=False):
    """
    RAG pipeline with extra features:
    - Returns answer, sources, confidence score, and optionally full context.
    """
    results = retriever.retrieve(query, top_k=top_k, score_threshold=min_score)
    if not results:
        return {'answer': 'No relevant context found.', 'sources': [], 'confidence': 0.0, 'context': ''}
    
    # Prepare context and sources
    context = "\n\n".join([doc['content'] for doc in results])
    sources = [{
        'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
        'page': doc['metadata'].get('page', 'unknown'),
        'score': doc['similarity_score'],
        'preview': doc['content'][:300] + '...'
    } for doc in results]
    confidence = max([doc['similarity_score'] for doc in results])
    
    # Generate answer
    prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {query}\n\nAnswer:"""
    response = llm.invoke([prompt.format(context=context, query=query)])
    
    output = {
        'answer': response.content,
        'sources': sources,
        'confidence': confidence
    }
    if return_context:
        output['context'] = context
    return output

# Example usage:
result = rag_advanced("what is Time-Series Forecasting for Predictive Resource Management?", rag_retriever, llm, top_k=3, min_score=0.0, return_context=True)
print("Answer:", result['answer'])
print("Sources:", result['sources'])
print("Confidence:", result['confidence'])
print("Context Preview:", result['context'][:300])

### Advanced RAG pipeline

In [36]:
# --- Advanced RAG Pipeline: Streaming, Citations, History, Summarization ---
from typing import List, Dict, Any
import time

class AdvancedRAGPipeline:
    def __init__(self, retriever, llm):
        self.retriever = retriever
        self.llm = llm
        self.history = []  # Store query history

    def query(self, question: str, top_k: int = 5, min_score: float = 0.2, stream: bool = False, summarize: bool = False) -> Dict[str, Any]:
        # Retrieve relevant documents
        results = self.retriever.retrieve(question, top_k=top_k, score_threshold=min_score)
        if not results:
            answer = "No relevant context found."
            sources = []
            context = ""
        else:
            context = "\n\n".join([doc['content'] for doc in results])
            sources = [{
                'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
                'page': doc['metadata'].get('page', 'unknown'),
                'score': doc['similarity_score'],
                'preview': doc['content'][:120] + '...'
            } for doc in results]
            # Streaming answer simulation
            prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {question}\n\nAnswer:"""
            if stream:
                print("Streaming answer:")
                for i in range(0, len(prompt), 80):
                    print(prompt[i:i+80], end='', flush=True)
                    time.sleep(0.05)
                print()
            response = self.llm.invoke([prompt.format(context=context, question=question)])
            answer = response.content

        # Add citations to answer
        citations = [f"[{i+1}] {src['source']} (page {src['page']})" for i, src in enumerate(sources)]
        answer_with_citations = answer + "\n\nCitations:\n" + "\n".join(citations) if citations else answer

        # Optionally summarize answer
        summary = None
        if summarize and answer:
            summary_prompt = f"Summarize the following answer in 2 sentences:\n{answer}"
            summary_resp = self.llm.invoke([summary_prompt])
            summary = summary_resp.content

        # Store query history
        self.history.append({
            'question': question,
            'answer': answer,
            'sources': sources,
            'summary': summary
        })

        return {
            'question': question,
            'answer': answer_with_citations,
            'sources': sources,
            'summary': summary,
            'history': self.history
        }

# Example usage:
adv_rag = AdvancedRAGPipeline(rag_retriever, llm)
result = adv_rag.query("what is Time-Series Forecasting for Predictive Resource Management?", top_k=3, min_score=0.0, stream=True, summarize=True)
print("\nFinal Answer:", result['answer'])
print("Summary:", result['summary'])
print("History:", result['history'][-1])

Retrieving documents for query: 'what is Time-Series Forecasting for Predictive Resource Management?'
Top K: 3, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00,  5.65it/s]

Generated embeddings with shape: (1, 384)
Retrieved 1 documents (after filtering)
Streaming answer:
Use the following context to answer the question concisely.
Context:
Project
 
2

:
 
Time-Series
 
Forecasting
 
for
 
Predictive
 
Resource
 
Management
 
Objective:
 
To
 
build
 
a
 
predictive
 
model
 
that
 
anticipates
 
future
 
soil
 
moisture
 
trends
 
to
 
optimize
 
irrigation
 
efficiency.
 
Workflow:
 
Curated
 
a
 
temporal
 
dataset
 
by
 
merging
 
real-time
 
sensor
 
streams
 
with
 
historical
 
5-year
 
weather
 
patterns
 
fetched
 
from
 
the
 
OpenWeather
 
API.
 
Developed
 
and
 
trained
 
a
 
Long
 
Short-Term
 
Memory
 
(LSTM)
 
neural
 
network
 
architecture
 
to
 
capture
 
seasonal
 
and
 
daily
 
non-linear
 
patterns
 
in
 
humidity
 
and
 
temperature.
 
Conducted
 
rigorous
 
Hyperparameter
 
Tuning
 
of
 
the
 
window
 
size
 
and
 
dropout
 
rates
 
to
 
prevent
 
the
 
model
 
from
 
overfitting
 
to
 
extreme
 
weather
 
anomalies.
 
Learning
 
Outcome:
  
handling
  
sequential
 
data
 
and
 
learned
 
to
 
use
 
Root
 
Mean
 
Squared
 
Error
 
(RMSE)
 
as
 
a
 
key
 
metric
 
for
 
validating
 
regression

Question: what i